# Hybrid search — BM25 + dense + RRF fusion

Dense retrieval is great at paraphrase but loses on exact entity / acronym matches. BM25 is the opposite. **Hybrid retrieval** runs both, then fuses the result lists with Reciprocal Rank Fusion (RRF) — a parameter-free combiner that consistently beats either alone on heterogeneous corpora.

We compare three retrievers on the canonical Q&A set:
1. **Dense only** (hash-embedding cosine) — the naive-RAG baseline.
2. **BM25 only** (`rank_bm25`).
3. **Hybrid via RRF** — fuse rank lists with `score = sum(1 / (k + rank_i))`.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import numpy as np
from rank_bm25 import BM25Okapi
from shared.embedders import cosine_topk, hash_embed
from shared.loaders import load_corpus, load_golden_qa

DOCS = load_corpus()
QA = [q for q in load_golden_qa() if q.source_ids]
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
print('docs:', len(DOCS), '  answerable Qs:', len(QA))

docs: 10   answerable Qs: 26


## Build the two indexes
Dense via hashing embedder; sparse via BM25Okapi over whitespace tokens.

In [3]:
import re
TOKEN_RE = re.compile(r'[A-Za-z0-9]+')
def tokenize(t: str) -> list[str]:
    return [w.lower() for w in TOKEN_RE.findall(t)]

DIMS, SEED = 256, 0
doc_vecs = hash_embed(doc_texts, dims=DIMS, seed=SEED)
bm25 = BM25Okapi([tokenize(t) for t in doc_texts])
print('dense:', doc_vecs.shape, '  sparse: BM25Okapi over', len(doc_texts), 'docs')

dense: (10, 256)   sparse: BM25Okapi over 10 docs


## Three retrieve functions

In [4]:
def retrieve_dense(q: str, k: int = 5) -> list[tuple[str, float]]:
    qv = hash_embed([q], dims=DIMS, seed=SEED)[0]
    idx, scores = cosine_topk(qv, doc_vecs, k=k)
    return [(doc_ids[i], float(s)) for i, s in zip(idx, scores)]

def retrieve_bm25(q: str, k: int = 5) -> list[tuple[str, float]]:
    scores = bm25.get_scores(tokenize(q))
    order = np.argsort(-scores)[:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

def retrieve_hybrid_rrf(q: str, k: int = 5, rrf_k: int = 60) -> list[tuple[str, float]]:
    dense = [doc for doc, _ in retrieve_dense(q, k=len(doc_ids))]
    sparse = [doc for doc, _ in retrieve_bm25(q, k=len(doc_ids))]
    fused: dict[str, float] = {}
    for rank, doc in enumerate(dense):
        fused[doc] = fused.get(doc, 0.0) + 1.0 / (rrf_k + rank)
    for rank, doc in enumerate(sparse):
        fused[doc] = fused.get(doc, 0.0) + 1.0 / (rrf_k + rank)
    ordered = sorted(fused.items(), key=lambda kv: -kv[1])[:k]
    return ordered

## Sanity check on one entity-heavy query
BM25 should crush dense on a query that asks for an exact identifier.

In [5]:
q = 'What is the routing strategy in RA-MoE?'
print('dense :', retrieve_dense(q, k=3))
print('bm25  :', retrieve_bm25(q, k=3))
print('hybrid:', retrieve_hybrid_rrf(q, k=3))

dense : [('synth-001', 0.23276573419570923), ('synth-003', 0.15188169479370117), ('synth-007', 0.11712891608476639)]
bm25  : [('synth-001', 9.552148900305594), ('synth-008', 1.157196950243617), ('synth-005', 1.0981026947826227)]
hybrid: [('synth-001', 0.03333333333333333), ('synth-008', 0.032266458495966696), ('synth-003', 0.03177805800756621)]


## Recall sweep across the golden Q&A
For each retriever, count how many questions have at least one source in the top-k.

In [6]:
def recall_at(retriever, k: int) -> float:
    hits = 0
    for item in QA:
        got = {doc for doc, _ in retriever(item.question, k=k)}
        if set(item.source_ids) & got:
            hits += 1
    return round(hits / len(QA), 4)

import json
table = {}
for name, fn in [('dense', retrieve_dense), ('bm25', retrieve_bm25), ('hybrid_rrf', retrieve_hybrid_rrf)]:
    table[name] = {f'recall@{k}': recall_at(fn, k) for k in (1, 3, 5)}
print(json.dumps(table, indent=2))

{
  "dense": {
    "recall@1": 0.6923,
    "recall@3": 0.8846,
    "recall@5": 0.9615
  },
  "bm25": {
    "recall@1": 0.9231,
    "recall@3": 1.0,
    "recall@5": 1.0
  },
  "hybrid_rrf": {
    "recall@1": 0.8846,
    "recall@3": 0.9615,
    "recall@5": 1.0
  }
}


## When does hybrid help most?
* Corpora with **acronyms, IDs, exact names** that dense embedders lose.
* Mixed query styles — some users type exact entities, others paraphrase intent.
* Multilingual settings where one side handles morphology better than the other.

## When it doesn't
* Pure-paraphrase corpora — BM25 contributes noise, hybrid <= dense.
* When the bottleneck is the *answer model*, not retrieval — fix the LLM prompt first.

Forward reference: [`04-reranking/`](../04-reranking/) is the cleanup step most hybrid pipelines need — fetch 50–100 with hybrid, rerank with a stronger model down to 5.